<img src="https://s3-us-west-2.amazonaws.com/public.notion-static.com/7fa59d58-ba42-4de1-840a-e2e31ab9ce3b/ba416d9e-ee4d-4b7a-a9fe-a506333af2b7.png" alt="Abstract Banner" width="100%" height="200" style="object-fit: cover; border-radius: 8px;">
by: Elmar Leonard, Muhammad Rafi Andrianto and Valencia

# **Section 0: Project Initialization**

## **0.1 Importing Library**

In [1]:
import pandas as pd
import numpy as np

import unicodedata

pd.set_option('display.max_columns', None)

## **0.2 Global Configuration**

In [2]:
DATA_DIR = 'Dataset' 

orders       = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_orders_dataset.csv')
customers    = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_customers_dataset.csv')
order_items  = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_order_items_dataset.csv')
payments     = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_order_payments_dataset.csv')
reviews      = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_order_reviews_dataset.csv')
products     = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_products_dataset.csv')
sellers      = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_sellers_dataset.csv')
geolocation  = pd.read_csv(f'../{DATA_DIR}/Raw Data/olist_geolocation_dataset.csv')
cat_translation = pd.read_csv(f'../{DATA_DIR}/Raw Data/product_category_name_translation.csv')

# **Section 1: Data Explanation**

**Dataset Overview**

* **Source File:** `Brazilian E-Commerce Public Dataset by Olist`
* **Total Dataset:** `9 csv`
* **Feature Dimensionality:** `52 columns across the csv`

## **1.1 General Information**

In [3]:
datasets = {
    'orders': orders,
    'customers': customers,
    'items': order_items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation': geolocation,
    'category_translation': cat_translation
}

for name, df in datasets.items():
    print("=" * 50)
    print(f" TABLE: {name.upper()} {df.shape}")
    print("=" * 50)
    
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    null_summary = pd.DataFrame({'Missing Values': missing, 'Percentage (%)': missing_pct})
    print(null_summary)
    
    if null_summary['Missing Values'].sum() == 0:
        print("No missing values found!")
        
    print("\nColumn Data Types:")
    print(df.dtypes)
    print("\n")

 TABLE: ORDERS (99441, 8)
                               Missing Values  Percentage (%)
order_id                                    0        0.000000
customer_id                                 0        0.000000
order_status                                0        0.000000
order_purchase_timestamp                    0        0.000000
order_approved_at                         160        0.160899
order_delivered_carrier_date             1783        1.793023
order_delivered_customer_date            2965        2.981668
order_estimated_delivery_date               0        0.000000

Column Data Types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


 TABLE: CUSTOMERS (99441, 5)
                          Missing Values  Percentage (%

In [4]:
total_transactions = customers['customer_id'].nunique()
unique_customers = customers['customer_unique_id'].nunique()

print(f"total transactions (customer_id): {total_transactions}")
print(f"total unique customers (customer_unique_id): {unique_customers}")
print(f"repeat customers count: {total_transactions - unique_customers}")
print(f"repeat customer ratio: {((total_transactions - unique_customers) / total_transactions) * 100:.2f}%")

total transactions (customer_id): 99441
total unique customers (customer_unique_id): 96096
repeat customers count: 3345
repeat customer ratio: 3.36%


In [5]:
print("multi order sample")
multi_item_id = order_items.groupby('order_id').filter(lambda x: len(x) > 1)['order_id'].iloc[3]
display(order_items[order_items['order_id'] == multi_item_id][['order_id', 'order_item_id', 'product_id', 'price', 'freight_value']])

print("multi payment sample")
multi_pay_id = payments.groupby('order_id').filter(lambda x: len(x) > 1)['order_id'].iloc[0]
display(payments[payments['order_id'] == multi_pay_id][['order_id', 'payment_sequential', 'payment_type', 'payment_value']])

multi order sample


,order_id,order_item_id,product_id,price,freight_value
32,00143d0f86d6fbd9f9b38ab440ac16f5,1,e95ee6822b66ac6058e2e4aff656071a,21.33,15.1
33,00143d0f86d6fbd9f9b38ab440ac16f5,2,e95ee6822b66ac6058e2e4aff656071a,21.33,15.1
34,00143d0f86d6fbd9f9b38ab440ac16f5,3,e95ee6822b66ac6058e2e4aff656071a,21.33,15.1


multi payment sample


,order_id,payment_sequential,payment_type,payment_value
25,5cfd514482e22bc992e7693f0e3e8df7,2,voucher,45.17
57742,5cfd514482e22bc992e7693f0e3e8df7,1,credit_card,665.41


## **1.2 Data Information**

### **1.2.1 olist_orders_dat**


| Column | Type | Description |
| --- | --- | --- |
| order_id | string | Unique identifier of the order (an order can have multiple items) |
| customer_id | string | Key to the customer who placed the order. |
| order_status | string | Order lifecycle status (delivered, shipped, canceled, etc). |
| order_purchase_timestamp | datetime (string) | Timestamp when the order was placed. |
| order_approved_at | datetime (string) | Timestamp when payment was approved. |
| order_delivered_carrier_date | datetime (string) | Timestamp when the order was handed to the logistics carrier. |
| order_delivered_customer_date | datetime (string) | Timestamp when the order was actually delivered to the customer. |
| order_estimated_delivery_date | datetime (string) | Delivery date estimate given to the customer at purchase time. |

### **1.2.2 olist_customers_dat**

| Column | Type | Description |
| --- | --- | --- |
| customer_id | string | key to the orders dataset. Each order has a unique customer_id. |
| customer_unique_id | string | The actual unique person (use this to track customers across multiple orders) |
| customer_zip_code_prefix | integer | First 5 digits of customer's zip code. |
| customer_city | string | Customer's city. |
| customer_state | string | Customer's state (Brazilian state code, e.g. SP, RJ). |

### **1.2.3 olist_order_items_dat**

| Column | Type | Description |
| --- | --- | --- |
| order_id | string | Unique identifier of the order. |
| order_item_id | integer | Sequential number identifying number of items within the order (1, 2, 3...). |
| product_id | string | Product unique identifier. |
| seller_id | string | Seller unique identifier. |
| shipping_limit_date | datetime (string) | Seller's deadline to hand the item to the logistic partner (carrier). |
| price | float | Item price. |
| freight_value | float | Freight/shipping cost for this item. (if an order has more than one item the freight value is splitted between items) |

### **1.2.4 olist_order_payments_dat**

| Column | Type | Description |
| --- | --- | --- |
| order_id | string | Unique identifier of the order. |
| payment_sequential | integer | Sequence number if a customer used more than one payment method for the same order. |
| payment_type | string | Payment method (credit_card, boleto, voucher, debit_card). |
| payment_installments | integer | Number of installments chosen. |
| payment_value | float | Amount paid via this payment row. |

### **1.2.5 olist_order_reviews_dat**

| Column | Type | Description |
| --- | --- | --- |
| review_id | string | Unique identifier of the review. |
| order_id | string | Unique identifier of the order. |
| review_score | integer | Customer's satisfaction rating, 1 (worst) to 5 (best). |
| review_comment_title | string | Title of the written review left by the customer (in portuguese). |
| review_comment_message | string | Comment message left by the customer (in portuguese). |
| review_creation_date | datetime (string) | Date the satisfaction survey was sent to the customer. |
| review_answer_timestamp | datetime (string) | Timestamp when the customer submitted the review. |

### **1.2.6 olist_products_dat**

| Column | Type | Description |
| --- | --- | --- |
| product_id | string | Unique identifier of the product. |
| product_category_name | string | Product category name  (in portuguese). |
| product_name_lenght | float | Number of characters in the product name. |
| product_description_lenght | float | Number of characters in the product description. |
| product_photos_qty | float | Number of photos published for the product. |
| product_weight_g | float | Product weight in grams. |
| product_length_cm | float | Product length in cm. |
| product_height_cm | float | Product height in cm. |
| product_width_cm | float | Product width in cm. |

### **1.2.7 olist_sellers_dat**

| Column | Type | Description |
| --- | --- | --- |
| seller_id | string | Unique identifier of the seller. |
| seller_zip_code_prefix | integer | First 5 digits of seller's zip code. |
| seller_city | string | Seller's city. |
| seller_state | string | Seller's state. |

### **1.2.8 product_category_name_translation**

| Column | Type | Description |
| --- | --- | --- |
| product_category_name | string | Category name in Portuguese. |
| product_category_name_english | string | Same category translated to English. |

### **1.2.9 olist_geolocation_dataset**

| Column | Type | Description |
| --- | --- | --- |
| geolocation_zip_code_prefix | integer | First 5 digits of the zip code. |
| geolocation_lat | float | Latitude of the zip code location. |
| geolocation_lng | float | Longitude of the zip code location. |
| geolocation_city | string | City name for the zip code. |
| geolocation_state | string | State abbreviation for the zip code. |

# **Section 2: Data Merging**

## **2.1 First Merge (orders + order_items)**

Joining `olist_orders_dataset` with `olist_order_items_dataset` expands our dataset from 99,441 rows to 113,425 rows, allowing us to analyze individual product prices and freight charges.

In [6]:
# order_items
df = orders.merge(order_items, on='order_id', how='left')
print('+ order_items: ', df.shape)

display(df.head())

+ order_items:  (113425, 14)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


## **2.2 Product Details & Translation (df + products & cat_translation)**

Joining with `olist_products_dataset` attaches product physical dimensions (weight, length, height, width) and translates Portuguese product category names into English from `product_category_name_translation`.

In [7]:
# products ( & translate category to English)
translate = products.merge(cat_translation, on='product_category_name', how='left')

df = df.merge(translate, on='product_id', how='left')
print('+ products: ', df.shape)
display(df.head())


+ products:  (113425, 23)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery


## **2.3 Seller Metadata (df + sellers)**

Adds seller geographic attributes (seller_city, seller_state, seller_zip_code_prefix).

In [8]:
# sellers
df = df.merge(sellers, on='seller_id', how='left')
print('+ sellers: ', df.shape)
display(df.head())

+ sellers:  (113425, 26)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP


## **2.4 Customer Data (df + customers)**

Maps the transaction token (customer_id) to the persistent buyer ID (customer_unique_id) and attaches buyer location details (customer_city, customer_state).

In [9]:
# customers
df = df.merge(customers, on='customer_id', how='left')
print('+ customers: ', df.shape)
display(df.head())

+ customers:  (113425, 30)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


## **2.5 Payments Merge (df + payments_agg)**

Orders can support multiple payment methods, such as splitting costs between credit cards and vouchers. To verify if an order uses multiple primary payment types or just a single voucher, the following test will be conducted:

In [10]:
type_sets = payments.groupby('order_id')['payment_type'].apply(lambda s: set(s))
non_voucher_types = type_sets.apply(lambda s: s - {'voucher'})
non_voucher_count = non_voucher_types.apply(len)

print(non_voucher_count.value_counts())

outliers = non_voucher_count[non_voucher_count >= 2]
print(f'\norders breaking the pattern: {len(outliers)}')
if len(outliers) > 0:
    print(type_sets[outliers.index])

payment_type
1    97818
0     1621
2        1
Name: count, dtype: int64

orders breaking the pattern: 1
order_id
a079628ac8002126e75f86b0f87332e4    {credit_card, debit_card}
Name: payment_type, dtype: object


The test results indicate that `99.99%` of orders used only one major payment type alongside a voucher, or relied entirely on a voucher. Consequently, we can conclude that the main program only supports a single primary payment system. This conclusion is backed by the functionality of the Olist website, which restricts users to selecting one main payment system, with the option to supplement or fulfill the payment using vouchers.  

Based on these findings, we can now proceed to merge the payments dataset. In this step, we will extract the primary payment system, aggregate the total number of vouchers used per order, and calculate the total value contributed by those vouchers.

In [11]:
def _agg_payments(g):
    g = g.sort_values('payment_sequential')
    non_voucher = g[g['payment_type'] != 'voucher']
    voucher_rows = g[g['payment_type'] == 'voucher']

    if len(non_voucher) > 0:
        dominant_type = non_voucher.groupby('payment_type')['payment_value'].sum().idxmax()
        dominant_installments = non_voucher.loc[non_voucher['payment_type'] == dominant_type, 'payment_installments'].max()
    else:
        dominant_type = 'voucher'
        dominant_installments = g['payment_installments'].max()

    return pd.Series({
        'total_payment_value': g['payment_value'].sum(),
        'payment_installments': dominant_installments,
        'payment_type': dominant_type,
        'n_vouchers': len(voucher_rows),
        'voucher_value': voucher_rows['payment_value'].sum()
    })

payments_agg = payments.groupby('order_id').apply(_agg_payments).reset_index()

df = df.merge(payments_agg, on='order_id', how='left')
print('+ payments (agg): ', df.shape)
display(df.head())

+ payments (agg):  (113425, 35)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,1.0,credit_card,2.0,20.59
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.0,boleto,0.0,0.00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,3.0,credit_card,0.0,0.00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.0,credit_card,0.0,0.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.0,credit_card,0.0,0.00


## **2.6 Deduplicated Reviews (df + reviews_dedup)**

`551 duplicate` rows across `547 orders (0.56%)` in olist_order_reviews_dataset contain multiple review submissions. Merging reviews directly without handling these duplicate order IDs would multiply those item rows in the final dataset. Sorting chronologically by review_answer_timestamp and keeping the latest record will result in a strict 1-to-1 join per order while retaining the customer's final updated rating.

In [12]:
# reviews: order_id should be ~unique, but merge safely
reviews_dedup = reviews.sort_values('review_answer_timestamp').drop_duplicates('order_id', keep='last')

df = df.merge(reviews_dedup[['order_id', 'review_score', 'review_comment_message',
                              'review_creation_date', 'review_answer_timestamp']],
              on='order_id', how='left')
              
print('+ reviews: ', df.shape)
display(df.head())

+ reviews:  (113425, 39)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,1.0,credit_card,2.0,20.59,4.0,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.0,boleto,0.0,0.00,4.0,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,3.0,credit_card,0.0,0.00,5.0,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.0,credit_card,0.0,0.00,5.0,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.0,credit_card,0.0,0.00,5.0,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51


## **2.6 Seller & Customer Geolocation (df + geo_seller + geo_customer)**

The geolocation dataset consists of `1,001,163 rows` containing multiple coordinate entries for each unique zip code. To establish a single, definitive latitude and longitude for each zip code, the data must undergo a cleaning process. First, the dataset will be filtered to retain only the coordinates that fall within Brazil's geographic boundaries. Afterward, the remaining latitude and longitude values will be aggregated to calculate the centroid for each unique zip code.

In [13]:
# geolocation
geolocation = geolocation[(geolocation['geolocation_lat'] >= -33.750833) & (geolocation['geolocation_lat'] <= 5.269444) &
                          (geolocation['geolocation_lng'] >= -73.983056) & (geolocation['geolocation_lng'] <= -34.793056)]

def _mode_or_first(s):
    m = s.mode()
    return m.iat[0] if not m.empty else s.iloc[0]

geolocation = geolocation.groupby('geolocation_zip_code_prefix', as_index=False).agg(
    geolocation_lat=('geolocation_lat', 'mean'),
    geolocation_lng=('geolocation_lng', 'mean'),
    geolocation_city=('geolocation_city', _mode_or_first),
    geolocation_state=('geolocation_state', _mode_or_first),
)

df['seller_zip_code_prefix'] = df['seller_zip_code_prefix'].astype('Int64')
df['customer_zip_code_prefix'] = df['customer_zip_code_prefix'].astype('Int64')
geolocation['geolocation_zip_code_prefix'] = geolocation['geolocation_zip_code_prefix'].astype('Int64')

The `geo_seller` dataset was created by renaming geolocation columns to prevent naming conflicts with the customer geolocation data, which will be merged later.

In [14]:
# seller geo
geo_seller = geolocation.rename(columns={
    'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
    'geolocation_lat': 'seller_lat',
    'geolocation_lng': 'seller_lng',
    'geolocation_city': 'seller_geo_city',
    'geolocation_state': 'seller_geo_state',
})
df = df.merge(geo_seller, on='seller_zip_code_prefix', how='left')
print('+ geolocation (seller): ', df.shape)

+ geolocation (seller):  (113425, 43)


Similarly to `geo_seller`, the `geo_customer` dataset was created by renaming its geolocation columns to prevent naming conflicts with the seller geolocation data that was merged previously.

In [15]:
# customer geo
geo_customer = geolocation.rename(columns={
    'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
    'geolocation_lat': 'customer_lat',
    'geolocation_lng': 'customer_lng',
    'geolocation_city': 'customer_geo_city',
    'geolocation_state': 'customer_geo_state',
})
df = df.merge(geo_customer, on='customer_zip_code_prefix', how='left')
print('+ geolocation (customer): ', df.shape)

display(df.head())

+ geolocation (customer):  (113425, 47)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350,maua,SP,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,1.0,credit_card,2.0,20.59,4.0,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,-23.680729,-46.444238,maua,SP,-23.576983,-46.587161,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570,belo horizonte,SP,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.0,boleto,0.0,0.00,4.0,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,-19.807681,-43.980427,belo horizonte,MG,-12.177924,-44.660711,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840,guariba,SP,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,3.0,credit_card,0.0,0.00,5.0,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,-21.363502,-48.229601,guariba,SP,-16.745150,-48.514783,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842,belo horizonte,MG,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.0,credit_card,0.0,0.00,5.0,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58,-19.837682,-43.924053,belo horizonte,MG,-5.774190,-35.271143,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752,mogi das cruzes,SP,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.0,credit_card,0.0,0.00,5.0,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51,-23.543395,-46.262086,mogi das cruzes,SP,-23.676370,-46.514627,santo andre,SP


## **2.7 Exporting Merged Dataset**

In [16]:
df.to_csv(f'../{DATA_DIR}/Merged Data/merged.csv', index=False)
print('Saved with shape', df.shape)

Saved with shape (113425, 47)


# **Section 3: Data Cleaning**

## **3.1 Structural Integrity**

### **3.1.1 Data Cleaning**

In [17]:
df.duplicated().sum()

np.int64(0)

The preliminary data quality check verified that the dataset contains zero redundant or duplicated entries.

In [18]:
df.duplicated(subset="order_id").sum()

np.int64(13984)

In [19]:
df[df.duplicated(subset="order_id", keep=False)].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
10,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,delivered,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07 00:00:00,1.0,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,2017-05-22 19:50:18,99.0,30.53,ferramentas_jardim,36.0,450.0,1.0,9000.0,42.0,12.0,39.0,garden_tools,29156,cariacica,ES,f2a85dec752b8517b5e58a06ff3cd937,20780,rio de janeiro,RJ,259.06,1.0,credit_card,0.0,0.0,1.0,Aguardando retorno da loja,2017-05-30 00:00:00,2017-05-30 23:13:47,-20.278513,-40.411675,cariacica,ES,-22.896463,-43.272172,rio de janeiro,RJ
11,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,delivered,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07 00:00:00,2.0,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,2017-05-22 19:50:18,99.0,30.53,ferramentas_jardim,36.0,450.0,1.0,9000.0,42.0,12.0,39.0,garden_tools,29156,cariacica,ES,f2a85dec752b8517b5e58a06ff3cd937,20780,rio de janeiro,RJ,259.06,1.0,credit_card,0.0,0.0,1.0,Aguardando retorno da loja,2017-05-30 00:00:00,2017-05-30 23:13:47,-20.278513,-40.411675,cariacica,ES,-22.896463,-43.272172,rio de janeiro,RJ
27,acce194856392f074dbf9dada14d8d82,7e20bf5ca92da68200643bda76c504c6,delivered,2018-06-04 00:00:13,2018-06-05 00:35:10,2018-06-05 13:24:00,2018-06-16 15:20:55,2018-07-18 00:00:00,1.0,d70f38e7f79c630f8ea00c993897042c,977f9f63dd360c2a32ece2f93ad6d306,2018-06-13 00:35:10,90.9,48.64,bebes,53.0,233.0,1.0,10950.0,41.0,40.0,40.0,baby,14910,tabatinga,SP,576ea0cab426cd8a00fad9a9c90a4494,41213,salvador,BA,227.68,10.0,credit_card,0.0,0.0,1.0,Até o momento não recebi o produto Protetor De...,2018-06-17 00:00:00,2018-06-20 11:38:22,-21.737063,-48.687601,tabatinga,SP,-12.939062,-38.438353,salvador,BA
28,acce194856392f074dbf9dada14d8d82,7e20bf5ca92da68200643bda76c504c6,delivered,2018-06-04 00:00:13,2018-06-05 00:35:10,2018-06-05 13:24:00,2018-06-16 15:20:55,2018-07-18 00:00:00,2.0,9451e630d725c4bb7a5a206b48b99486,d673a59aac7a70d8b01e6902bf090a11,2018-06-13 00:35:10,39.5,48.64,bebes,52.0,300.0,1.0,350.0,31.0,10.0,12.0,baby,14940,ibitinga,SP,576ea0cab426cd8a00fad9a9c90a4494,41213,salvador,BA,227.68,10.0,credit_card,0.0,0.0,1.0,Até o momento não recebi o produto Protetor De...,2018-06-17 00:00:00,2018-06-20 11:38:22,-21.757321,-48.829744,ibitinga,SP,-12.939062,-38.438353,salvador,BA
54,9faeb9b2746b9d7526aef5acb08e2aa0,79183cd650e2bb0d475b0067d45946ac,delivered,2018-07-26 14:39:59,2018-07-26 14:55:10,2018-07-27 12:04:00,2018-07-31 22:26:55,2018-08-16 00:00:00,1.0,f48eb5c2fde13ca63664f0bb05f55346,f7ba60f8c3f99e7ee4042fdef03b70c4,2018-07-30 14:55:10,60.0,15.52,esporte_lazer,60.0,1153.0,2.0,100.0,20.0,11.0,11.0,sports_leisure,9628,sao bernardo do campo,SP,c77154776ead8e798c2d684205938f71,90620,porto alegre,RS,151.04,2.0,credit_card,0.0,0.0,1.0,"Recebi apenas 1 unidade solicitada, deveriam s...",2018-08-01 00:00:00,2018-08-04 02:14:45,-23.661305,-46.564296,sao bernardo do campo,SP,-30.049925,-51.201923,porto alegre,RS


While the dataset contains **13,984** rows sharing a duplicate `order_id`, this is expected and valid, not a data quality problem. It reflects **multi-item orders** where an order contains more than one product, each item gets its own row, so the same `order_id` appears once per item. To preserve this purchase-level detail (price, product, and seller can differ per item within one order), these rows are kept as-is rather than deduplicated.

### **3.1.2 Missing Data**

#### **3.1.2.1 General Missing Data based on Order Status**

In [20]:
print(f"Missing Data Outside of Delivered Status")
print(f"Total Rows  = {df[df["order_status"]!="delivered"].shape[0]}")
display(df[df["order_status"]!="delivered"].isnull().sum().to_frame().T)

Missing Data Outside of Delivered Status
Total Rows  = 3228


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,146,1966,3221,0,775,775,775,775,775,775,841,841,841,841,775,775,775,775,843,775,775,775,0,0,0,0,0,0,0,0,0,134,1203,134,134,779,779,779,779,18,18,18,18


In [21]:
print(f"Missing Data with Delivered Status")
print(f"Total Rows  = {df[df["order_status"]=="delivered"].shape[0]}")
display(df[df["order_status"]=="delivered"].isnull().sum().to_frame().T)

Missing Data with Delivered Status
Total Rows  = 110197


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,15,2,8,0,0,0,0,0,0,0,1537,1537,1537,1537,18,18,18,18,1559,0,0,0,0,0,0,0,3,3,3,3,3,827,64301,827,827,249,249,249,249,289,289,289,289


After comparing the two missing value summaries above, it shows that the missing data is heavily concentrated in orders whose `order_status` is **not** `delivered`. non-delivered orders carry far more nulls across nearly every column than delivered ones do. This makes complete sense because an order that was canceled, in transit, or never shipped simply never generates some of these fields (e.g. a delivery date) in the first place, since that stage of the order never happened.

The sections below work through each missing data cluster individually, starting with the seller/customer location gaps, which turn out to be the single largest contributor.

#### **3.1.2.2 Fixing Seller and Customer Location Information**

The missing values in the seller and customer location data occurred because several zip codes were omitted from the source geolocation dataset. Consequently, during the data merging process, the absence of these zip codes in the reference data resulted in NaN values within those columns.

In [22]:
add_loc = pd.read_csv(f"../Dataset/Supporting Data/brazil_zip_codes_coordinates.csv")

In [23]:
ref_indexed = add_loc.set_index('Zip Code (CEP)')

main_subset = df[['customer_zip_code_prefix', 'customer_geo_city', 'customer_geo_state', 'customer_lat', 'customer_lng']].set_index('customer_zip_code_prefix')

ref_indexed = ref_indexed.rename(columns={
    'City': 'customer_geo_city',
    'State': 'customer_geo_state',
    'Centroid Latitude': 'customer_lat',
    'Centroid Longitude': 'customer_lng'
})

main_subset.update(ref_indexed)

df[['customer_geo_city', 'customer_geo_state', 'customer_lat', 'customer_lng']] = main_subset.values


In [24]:
print(f"Missing Data Outside of Delivered Status")
print(f"Total Rows  = {df[df["order_status"]!="delivered"].shape[0]}")
display(df[df["order_status"]!="delivered"].isnull().sum().to_frame().T)

Missing Data Outside of Delivered Status
Total Rows  = 3228


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,146,1966,3221,0,775,775,775,775,775,775,841,841,841,841,775,775,775,775,843,775,775,775,0,0,0,0,0,0,0,0,0,134,1203,134,134,779,779,779,779,0,0,0,0


In [25]:
print(f"Missing Data with Delivered Status")
print(f"Total Rows  = {df[df["order_status"]=="delivered"].shape[0]}")
display(df[df["order_status"]=="delivered"].isnull().sum().to_frame().T)

Missing Data with Delivered Status
Total Rows  = 110197


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,15,2,8,0,0,0,0,0,0,0,1537,1537,1537,1537,18,18,18,18,1559,0,0,0,0,0,0,0,3,3,3,3,3,827,64301,827,827,249,249,249,249,0,0,0,0


In [26]:
ref_indexed = add_loc.set_index('Zip Code (CEP)')

main_subset = df[['seller_zip_code_prefix', 'seller_geo_city', 'seller_geo_state', 'seller_lat', 'seller_lng']].set_index('seller_zip_code_prefix')

ref_indexed = ref_indexed.rename(columns={
    'City': 'seller_geo_city',
    'State': 'seller_geo_state',
    'Centroid Latitude': 'seller_lat',
    'Centroid Longitude': 'seller_lng'
})

main_subset.update(ref_indexed)

df[['seller_geo_city', 'seller_geo_state', 'seller_lat', 'seller_lng']] = main_subset.values

In [27]:
print(f"Missing Data Outside of Delivered Status")
print(f"Total Rows  = {df[df["order_status"]!="delivered"].shape[0]}")
display(df[df["order_status"]!="delivered"].isnull().sum().to_frame().T)

Missing Data Outside of Delivered Status
Total Rows  = 3228


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,146,1966,3221,0,775,775,775,775,775,775,841,841,841,841,775,775,775,775,843,775,775,775,0,0,0,0,0,0,0,0,0,134,1203,134,134,775,775,775,775,0,0,0,0


In [28]:
print(f"Missing Data with Delivered Status")
print(f"Total Rows  = {df[df["order_status"]=="delivered"].shape[0]}")
display(df[df["order_status"]=="delivered"].isnull().sum().to_frame().T)

Missing Data with Delivered Status
Total Rows  = 110197


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,15,2,8,0,0,0,0,0,0,0,1537,1537,1537,1537,18,18,18,18,1559,0,0,0,0,0,0,0,3,3,3,3,3,827,64301,827,827,0,0,0,0,0,0,0,0


Although the primary missing values have been addressed, certain rows remain incomplete. Upon further inspection, these remaining gaps are linked to other seller-specific data points. This correlation will be analyzed in detail in the following section.

#### **3.1.2.3 Fixing Payment Information**

In [29]:
df[df["payment_type"].isnull()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
35037,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04 00:00:00,1.0,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,beleza_saude,34.0,1036.0,1.0,1000.0,16.0,16.0,16.0,health_beauty,81810,curitiba,PR,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,NaN,NaN,NaN,NaN,NaN,1.0,nao recebi o produto e nem resposta da empresa,2016-10-06 00:00:00,2016-10-07 18:32:28,-25.507014,-49.275963,curitiba,PR,-20.585751,-47.863693,sao joaquim da barra,SP
35038,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04 00:00:00,2.0,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,beleza_saude,34.0,1036.0,1.0,1000.0,16.0,16.0,16.0,health_beauty,81810,curitiba,PR,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,NaN,NaN,NaN,NaN,NaN,1.0,nao recebi o produto e nem resposta da empresa,2016-10-06 00:00:00,2016-10-07 18:32:28,-25.507014,-49.275963,curitiba,PR,-20.585751,-47.863693,sao joaquim da barra,SP
35039,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04 00:00:00,3.0,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,beleza_saude,34.0,1036.0,1.0,1000.0,16.0,16.0,16.0,health_beauty,81810,curitiba,PR,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,NaN,NaN,NaN,NaN,NaN,1.0,nao recebi o produto e nem resposta da empresa,2016-10-06 00:00:00,2016-10-07 18:32:28,-25.507014,-49.275963,curitiba,PR,-20.585751,-47.863693,sao joaquim da barra,SP


In [30]:
print(f"Percentage of Order ID in which have no payment information : {(len(df[df["payment_type"].isnull()]["order_id"].unique())/len(df["order_id"].unique()))*100:.3f}%")

Percentage of Order ID in which have no payment information : 0.001%


Standard operational protocols prevent an e-commerce platform like **Olist** from processing or delivering an order prior to payment confirmation. This means a missing `payment_type` for an otherwise processed order is not a real business state. These data anomaly likely stem from data truncation or errors during the export process. Specifically, the payment details for these orders may have been inadvertently dropped during the data masking phase.  
  
Because these specific anomalies affect only a single order ID or **0.001%** of the data, we have decided to drop this record from the dataset because its the safest choice rather than attempt to guess or impute payment.

In [31]:
df = df[df["order_id"]!="bfbd0f9bdef84302105ad712db648a6c"]


In [32]:
print(f"Missing Data with Delivered Status")
print(f"Total Rows  = {df[df["order_status"]=="delivered"].shape[0]}")
display(df[df["order_status"]=="delivered"].isnull().sum().to_frame().T)

Missing Data with Delivered Status


Total Rows  = 110194


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,15,2,8,0,0,0,0,0,0,0,1537,1537,1537,1537,18,18,18,18,1559,0,0,0,0,0,0,0,0,0,0,0,0,827,64301,827,827,0,0,0,0,0,0,0,0


#### **3.1.2.4 Fixing Order Approval & Delivery Date Information on Status Order Delivered**

In [33]:
df_summary = pd.DataFrame({
    "order_approved_at" : [df[(df["order_status"]=="delivered")&(df["order_approved_at"].isnull())]["order_id"].nunique()],
    "order_delivered_carrier_date" : [df[(df["order_status"]=="delivered")&(df["order_delivered_carrier_date"].isnull())]["order_id"].nunique()],
    "order_delivered_customer_date" : [df[(df["order_status"]=="delivered")&(df["order_delivered_customer_date"].isnull())]["order_id"].nunique()],
})

print(f"Total Unique Order ID that have Missing Value")
display(df_summary)

Total Unique Order ID that have Missing Value


,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
0,14,2,8


In [34]:
df[(df["order_status"]=="delivered")&(df["order_approved_at"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
6009,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00,1.0,0e20a07ca1714df21f9b07ca3bf7c682,4e7c18b98d84e05cbae3ff0ff03846c2,2017-02-22 13:40:00,309.90,39.11,eletroportateis,41.0,675.0,2.0,20800.0,75.0,40.0,40.0,small_appliances,14882,jaboticabal,SP,8a9a08c7ca8900a200d83cf838a07e0b,6708,cotia,SP,349.01,1.0,boleto,0.0,0.0,4.0,"Muita demora, mas, tudo ok.\nTerezinha",2017-03-21 00:00:00,2017-03-21 17:35:02,-21.24565,-48.317901,jaboticabal,SP,-23.592545,-46.847084,cotia,SP
18860,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00,1.0,2c2b6a28924791234bd386bddb17512e,a4b6b9b992b46e9ef863637af96e04bc,2017-02-22 11:45:31,379.00,17.86,construcao_ferramentas_seguranca,56.0,498.0,1.0,1008.0,33.0,14.0,26.0,construction_tools_safety,88090,florianopolis,SC,91efb7fcabc17925099dced52435837f,93548,novo hamburgo,RS,396.86,1.0,boleto,0.0,0.0,5.0,NaN,2017-03-03 00:00:00,2017-03-04 23:25:49,-27.592214,-48.595469,florianopolis,SC,-29.688885,-51.114436,novo hamburgo,RS
21661,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00,1.0,583916a5dae918f5e89baec139141c54,3481aa57cd91f9f9d3fa1fa12d9a3bf7,2017-02-22 12:29:47,49.99,15.53,NaN,NaN,NaN,NaN,3100.0,28.0,28.0,50.0,NaN,13360,capivari,SP,e1f01a1bd6485e58ad3c769a5427d8a8,8230,sao paulo,SP,65.52,1.0,boleto,0.0,0.0,5.0,NaN,2017-03-02 00:00:00,2017-03-03 07:21:03,-22.997484,-47.505725,capivari,SP,-23.523145,-46.45295,sao paulo,SP
25796,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00,1.0,c6dd917a0be2a704582055949915ab32,7a67c85e85bb2ce8582c35f2203ad736,2017-02-22 15:48:35,79.99,26.82,cool_stuff,54.0,1012.0,1.0,1200.0,42.0,25.0,15.0,cool_stuff,3426,sao paulo,SP,7e1a5ca61b572d76b64b6688b9f96473,62700,caninde,CE,106.81,1.0,boleto,0.0,0.0,5.0,NaN,2017-03-10 00:00:00,2017-03-13 00:18:46,-23.552336,-46.536869,sao paulo,SP,-4.355929,-39.314376,caninde,CE
26359,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00,1.0,c6dd917a0be2a704582055949915ab32,7a67c85e85bb2ce8582c35f2203ad736,2017-02-21 12:05:55,79.99,15.77,cool_stuff,54.0,1012.0,1.0,1200.0,42.0,25.0,15.0,cool_stuff,3426,sao paulo,SP,c8822fce1d0bfa7ddf0da24fff947172,27945,macae,RJ,95.76,1.0,boleto,0.0,0.0,5.0,NaN,2017-03-03 00:00:00,2017-03-04 00:06:38,-23.552336,-46.536869,sao paulo,SP,-22.372457,-41.794318,macae,RJ
30541,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00,1.0,5ab02ca028398131a5ae91401eb49788,80e6699fe29150b372a0c8a1ebf7dcc8,2017-01-23 12:48:08,39.99,14.52,esporte_lazer,33.0,1322.0,2.0,700.0,26.0,16.0,21.0,sports_leisure,83323,pinhais,PR,6ff8b0d7b35d5c945633b8d60165691b,11030,santos,SP,

In [35]:
df[(df["order_status"]=="delivered")&(df["order_delivered_carrier_date"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
83453,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00,1.0,44c2baf621113fa7ac95fa06b4afbc68,3f2af2670e104d1bcb54022274daeac5,2017-10-18 10:07:16,179.0,14.98,moveis_decoracao,54.0,984.0,2.0,7000.0,16.0,50.0,55.0,furniture_decor,87240,terra boa,PR,a2ac81ecc3704410ae240e74d4f0af40,13334,indaiatuba,SP,193.98,1.0,credit_card,0.0,0.0,5.0,NaN,2017-10-17 00:00:00,2017-10-17 10:56:02,-23.771858,-52.448328,terra boa,PR,-23.082208,-47.203433,indaiatuba,SP
105606,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00,1.0,30b5b5635a79548a48d04162d971848f,f9bbdd976532d50b7816d285a22bd01e,2017-06-04 23:30:16,179.0,15.00,esporte_lazer,43.0,1873.0,2.0,900.0,26.0,26.0,26.0,sports_leisure,5319,sao paulo,SP,d77cf4be2654aa70ef150f8bfec076a6,91330,porto alegre,RS,194.00,4.0,credit_card,0.0,0.0,5.0,NaN,2017-06-25 00:00:00,2017-06-27 01:49:04,-23.541812,-46.730687,sao paulo,SP,-30.033566,-51.16256,porto alegre,RS


In [36]:
df[(df["order_status"]=="delivered")&(df["order_delivered_customer_date"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
3376,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00,1.0,a50acd33ba7a8da8e9db65094fa990a4,8581055ce74af1daba164fdbd55a40de,2017-12-04 17:56:40,117.30,17.53,automotivo,53.0,555.0,1.0,4105.0,67.0,10.0,67.0,auto,7112,guarulhos,SP,13467e882eb3a701826435ee4424f2bd,18520,cerquilho,SP,134.83,3.0,credit_card,0.0,0.0,5.0,Chegou rápido tudo ok,2017-12-19 00:00:00,2017-12-19 04:15:39,-23.468704,-46.516142,guarulhos,SP,-23.166262,-47.746689,cerquilho,SP
23485,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00,1.0,2167c8f6252667c0eb9edd51520706a1,0bb738e4d789e63e2267697c42d35a2d,2018-06-26 07:19:05,329.00,25.24,industria_comercio_e_negocios,48.0,1581.0,1.0,7750.0,36.0,51.0,18.0,industry_commerce_and_business,18130,sao roque,SP,2f17c5b324ad603491521b279a9ff4de,18255,quadra,SP,354.24,1.0,debit_card,0.0,0.0,5.0,"Produto novo, muito bom.",2018-06-29 00:00:00,2018-06-29 16:26:37,-23.531424,-47.134428,sao roque,SP,-23.293915,-48.057538,quadra,SP
49966,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00,1.0,e7d5464b94c9a5963f7c686fc80145ad,58f1a6197ed863543e0136bdedb3fce2,2018-07-05 17:15:12,139.00,19.07,relogios_presentes,42.0,938.0,5.0,275.0,16.0,14.0,14.0,watches_gifts,36407,conselheiro lafaiete,MG,1bd06a0c0df8b23dacfd3725d2dc0bb9,12445,pindamonhangaba,SP,158.07,3.0,credit_card,0.0,0.0,5.0,NaN,2018-07-11 00:00:00,2018-07-11 19:27:46,-20.659212,-43.806927,conselheiro lafaiete,MG,-22.888249,-45.376194,pindamonhangaba,SP
90294,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00,1.0,e7d5464b94c9a5963f7c686fc80145ad,58f1a6197ed863543e0136bdedb3fce2,2018-07-05 22:15:14,139.00,19.07,relogios_presentes,42.0,938.0,5.0,275.0,16.0,14.0,14.0,watches_gifts,36407,conselheiro lafaiete,MG,3bc508d482a402715be4d5cf4020cc81,13170,sumare,SP,158.07,1.0,credit_card,0.0,0.0,5.0,NaN,2018-07-07 00:00:00,2018-07-10 11:38:13,-20.659212,-43.806927,conselheiro lafaiete,MG,-22.822137,-47.270335,sumare,SP
94388,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00,1.0,ec165cd31c50585786ffda6feff5d0a6,8bdd8e3fd58bafa48af76b2c5fd71974,2018-07-05 21:29:54,188.99,15.63,brinquedos,12.0,503.0,1.0,967.0,37.0,23.0,27.0,toys,1552,sao paulo,SP,ebf7e0d43a78c81991a4c59c145c75db,13560,sao carlos,SP,204.62,4.0,credit_card,0.0,0.0,5.0,O produto chegou muito antes do prazo previsto...,2018-07-06 00:00:00,2018-07-07 18:48:09,-23.569491,-46.611322,sao paulo,SP,-22.015004,-47.890181,sao carlos,SP
105606,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00,1.0,30b5b5635a79548a48d04162d971848f,f9bbdd976532d50b7816d285a22bd01e,2017-06-04 23:30:16,179.00,15.00,e

In [37]:
unique_orders = set()
unique_orders.update(df[(df["order_status"]=="delivered")&(df["order_approved_at"].isnull())]["order_id"].unique())
unique_orders.update(df[(df["order_status"]=="delivered")&(df["order_delivered_carrier_date"].isnull())]["order_id"].unique())
unique_orders.update(df[(df["order_status"]=="delivered")&(df["order_delivered_customer_date"].isnull())]["order_id"].unique())

print(f"Percentage of Order ID in which have no payment information : {(len(unique_orders)/len(df["order_id"].unique()))*100:.3f}%")

Percentage of Order ID in which have no payment information : 0.023%


An analysis of the data indicates that the missing values do not stem from operational failures, such as lost shipments or undelivered products. On the contrary, the products were successfully delivered, generally yielding positive customer reviews. The few negative reviews received were entirely unrelated to delivery issues, focusing instead on product quality or discrepancies in the items received.  
  
Furthermore, the affected rows represent a negligible **0.023%** of the total dataset. Removing these records is the safest approach, as attempting to impute them with artificial values could introduce bias and compromise the integrity of the final analysis.

In [38]:
df = df[~df['order_id'].isin(unique_orders)]

In [39]:
print(f"Missing Data with Delivered Status")
print(f"Total Rows  = {df[df["order_status"]=="delivered"].shape[0]}")
display(df[df["order_status"]=="delivered"].isnull().sum().to_frame().T)

Missing Data with Delivered Status
Total Rows  = 110170


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1536,1536,1536,1536,18,18,18,18,1558,0,0,0,0,0,0,0,0,0,0,0,0,827,64290,827,827,0,0,0,0,0,0,0,0


#### **3.1.2.5 Fixing Order Approval & Delivery Date Information on Status Order other than Delivered**

In [40]:
print(f"Missing Data Outside of Delivered Status")
print(f"Total Rows  = {df[df["order_status"]!="delivered"].shape[0]}")
display(df[df["order_status"]!="delivered"].isnull().sum().to_frame().T)

Missing Data Outside of Delivered Status
Total Rows  = 3228


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,146,1966,3221,0,775,775,775,775,775,775,841,841,841,841,775,775,775,775,843,775,775,775,0,0,0,0,0,0,0,0,0,134,1203,134,134,775,775,775,775,0,0,0,0


As concluded in Section 3.1.2.2 (Fixing Seller and Customer Location Information), the missing seller IDs share a strong correlation with the majority of missing values in non-delivered order statuses. Consequently, we will conduct a data validation check excluding these missing seller IDs. This will allow us to assess whether they heavily impact the missing data within the order approval and delivery date fields.

In [41]:
print(f"Missing Data Outside of Delivered Status")
print(f"Total Rows  = {df[df["order_status"]!="delivered"].shape[0]}")
display(df[(df["order_status"]!="delivered")&(df["seller_id"].notnull())].isnull().sum().to_frame().T)

Missing Data Outside of Delivered Status
Total Rows  = 3228


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,0,1192,2446,0,0,0,0,0,0,0,66,66,66,66,0,0,0,0,68,0,0,0,0,0,0,0,0,0,0,0,0,115,951,115,115,0,0,0,0,0,0,0,0


Even after excluding the missing `seller_id` rows, a substantial number of dates are still missing. `order_delivered_carrier_date` is missing **1,192** rows, and `order_delivered_customer_date` **2,446** rows. Since excluding the `seller_id` related nulls did not resolve this, the missing `seller_id` is not the explanation here so we instead look at how these missing dates break down by `order_status` to see if there is a different structural pattern underneath

In [42]:
df_summary = pd.DataFrame({
    "invoiced": [df[(df["order_status"]=="invoiced")&(df["seller_id"].notnull()&(df["order_delivered_carrier_date"].isnull()))]["order_id"].nunique()],
    "shipped": [df[(df["order_status"]=="shipped")&(df["seller_id"].notnull()&(df["order_delivered_carrier_date"].isnull()))]["order_id"].nunique()],
    "processing": [df[(df["order_status"]=="processing")&(df["seller_id"].notnull()&(df["order_delivered_carrier_date"].isnull()))]["order_id"].nunique()],
    "canceled": [df[(df["order_status"]=="canceled")&(df["seller_id"].notnull()&(df["order_delivered_carrier_date"].isnull()))]["order_id"].nunique()],
    "unavailable": [df[(df["order_status"]=="unavailable")&(df["seller_id"].notnull()&(df["order_delivered_carrier_date"].isnull()))]["order_id"].nunique()],
    "approved": [df[(df["order_status"]=="approved")&(df["seller_id"].notnull()&(df["order_delivered_carrier_date"].isnull()))]["order_id"].nunique()]
})

print("Total of Missing Values in Each Order Status")
display(df_summary)
total = df[(df["order_status"]!="delivered")&(df["seller_id"].notnull()&(df["order_delivered_carrier_date"].isnull()))]["order_id"].nunique()
print(f"Total of unique id for this missing value: {total}")
print(f"Total Percentage of unique id for this missing value: {(total/len(df[df["order_status"]!="delivered"]))*100:.2f}%")

Total of Missing Values in Each Order Status


,invoiced,shipped,processing,canceled,unavailable,approved
0,312,0,301,386,6,2


Total of unique id for this missing value: 1007
Total Percentage of unique id for this missing value: 31.20%


In [43]:
df_summary = pd.DataFrame({
    "invoiced": [df[(df["order_status"]=="invoiced")&(df["seller_id"].notnull()&(df["order_delivered_customer_date"].isnull()))]["order_id"].nunique()],
    "shipped": [df[(df["order_status"]=="shipped")&(df["seller_id"].notnull()&(df["order_delivered_customer_date"].isnull()))]["order_id"].nunique()],
    "processing": [df[(df["order_status"]=="processing")&(df["seller_id"].notnull()&(df["order_delivered_customer_date"].isnull()))]["order_id"].nunique()],
    "canceled": [df[(df["order_status"]=="canceled")&(df["seller_id"].notnull()&(df["order_delivered_customer_date"].isnull()))]["order_id"].nunique()],
    "unavailable": [df[(df["order_status"]=="unavailable")&(df["seller_id"].notnull()&(df["order_delivered_customer_date"].isnull()))]["order_id"].nunique()],
    "approved": [df[(df["order_status"]=="approved")&(df["seller_id"].notnull()&(df["order_delivered_customer_date"].isnull()))]["order_id"].nunique()]
})

print("Total of Missing Values in Each Order Status")
display(df_summary)
total = df[(df["order_status"]!="delivered")&(df["seller_id"].notnull()&(df["order_delivered_customer_date"].isnull()))]["order_id"].nunique()
print(f"Total of unique id for this missing value: {total}")
print(f"Total Percentage of unique id for this missing value: {(total/len(df[df["order_status"]!="delivered"]))*100:.2f}%")

Total of Missing Values in Each Order Status


,invoiced,shipped,processing,canceled,unavailable,approved
0,312,1106,301,455,6,2


Total of unique id for this missing value: 2182
Total Percentage of unique id for this missing value: 67.60%


In [44]:
df[(df["order_status"]!="delivered")&(df["seller_id"].notnull())&((df["order_delivered_customer_date"].isnull()|(df["order_delivered_carrier_date"].isnull())&(df["order_purchase_timestamp"].str.contains('2016'))))].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00,1.0,a1804276d9941ac0733cfd409f5206eb,dc8798cbf453b7e0f98745e396cc5616,2017-04-19 13:25:17,49.90,16.05,NaN,NaN,NaN,NaN,600.0,35.0,35.0,15.0,NaN,5455,sao paulo,SP,36edbb3fb164b1f16485364b6fb04c73,98900,santa rosa,RS,65.95,1.0,credit_card,0.0,0.0,2.0,fiquei triste por n ter me atendido.,2017-05-13 00:00:00,2017-05-13 20:25:42,-23.541383,-46.711854,sao paulo,SP,-27.865358,-54.470128,santa rosa,RS
46,ee64d42b8cf066f35eac1cf57de1aa85,caded193e8e47b8362864762a83db3c5,shipped,2018-06-04 16:44:48,2018-06-05 04:31:18,2018-06-05 14:32:00,NaN,2018-06-28 00:00:00,1.0,c50ca07e9e4db9ea5011f06802c0aea0,e9779976487b77c6d4ac45f75ec7afe9,2018-06-13 04:30:33,14.49,7.87,beleza_saude,59.0,1782.0,1.0,125.0,25.0,14.0,18.0,health_beauty,11701,praia grande,SP,08fb46d35bb3ab4037202c23592d1259,13215,jundiai,SP,22.36,1.0,boleto,0.0,0.0,1.0,NaN,2018-07-01 00:00:00,2018-07-11 20:41:18,-24.008923,-46.419125,praia grande,SP,-23.169668,-46.886715,jundiai,SP
118,0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03 17:44:42,2018-08-07 06:15:14,NaN,NaN,2018-08-21 00:00:00,1.0,1522589c64efd46731d3522568e5bc83,28405831a29823802aa22c084cfd0649,2018-08-13 06:15:14,35.00,15.35,artigos_de_natal,35.0,415.0,4.0,550.0,37.0,10.0,37.0,christmas_supplies,3644,sao paulo,SP,c7f8d7b1fffc946d7069574f74c39f4e,88140,santo amaro da imperatriz,SC,50.35,1.0,boleto,0.0,0.0,3.0,"Gostei do produto, porem fiquei preocupada não...",2018-08-25 00:00:00,2018-08-29 10:48:52,-23.526009,-46.533443,sao paulo,SP,-27.687415,-48.771889,santo amaro da imperatriz,SC
148,15bed8e2fec7fdbadb186b57c46c92f2,f3f0e613e0bdb9c7cee75504f0f90679,processing,2017-09-03 14:22:03,2017-09-03 14:30:09,NaN,NaN,2017-10-03 00:00:00,1.0,61d52f4882421048afd530db53d6f230,fa74b2f3287d296e9fbd2cc80f2d1cf1,2017-09-20 14:30:09,125.90,12.38,perfumaria,59.0,149.0,1.0,500.0,36.0,18.0,27.0,perfumery,19023,presidente prudente,SP,9f269af9c49244f6ba4a46985a3cfc2e,3436,sao paulo,SP,138.28,2.0,credit_card,0.0,0.0,5.0,NaN,2017-10-05 00:00:00,2017-10-05 12:55:11,-22.110681,-51.399092,presidente prudente,SP,-23.559514,-46.53141,sao paulo,SP
185,6942b8da583c2f9957e990d028607019,52006a9383bf149a4fb24226b173106f,shipped,2018-01-10 11:33:07,2018-01-11 02:32:30,2018-01-11 19:39:23,NaN,2018-02-07 00:00:00,1.0,ee0c1cf2fbeae95205b4aa506f1469f0,cc419e0650a3c5ba77189a1882b7556a,2018-01-18 02:32:30,53.99,15.13,perfumaria,44.0,334.0,1.0,200.0,16.0,16.0,13.0,perfumery,9015,santo andre,SP,528b011eb7fab3d59c336cc7248eed3a,38600,paracatu,MG,69.12,1.0,boleto,0.0,0.0,NaN,NaN,NaN,NaN,-23.659364,-46.523183,santo andre,SP,-17.224179,-46.874265,paracatu,MG


An evaluation of missing values across specific order statuses reveals a logical alignment with the e-commerce transactional lifecycle. Statuses like 'invoiced' (312 rows), 'processing' (301 rows), and 'shipped' (1,106 rows) inherently lack delivery and carrier timestamps because these transactions never reached final completion before the data snapshot was captured. These missing values are structural rather than accidental, representing unresolved operational states, logistics bottlenecks, or canceled lifecycles within the 2016–2018 timeframe.  
  
Furthermore, because these records account for a high percentage of their respective data categories, they will not be dropped. Instead, they will be retained in the dataset to preserve their utility for other areas of the analysis.

#### **3.1.2.6 Fixing Missing Seller Information**

In [45]:
print(f"Missing Data Outside of Delivered Status")
print(f"Total Rows  = {df[df["order_status"]!="delivered"].shape[0]}")
display(df[(df["order_status"]!="delivered")&(df["seller_id"].isnull())].isnull().sum().to_frame().T)

Missing Data Outside of Delivered Status
Total Rows  = 3228


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,146,774,775,0,775,775,775,775,775,775,775,775,775,775,775,775,775,775,775,775,775,775,0,0,0,0,0,0,0,0,0,19,252,19,19,775,775,775,775,0,0,0,0


In [46]:
df[(df["order_status"]!="delivered")&(df["seller_id"].isnull())&(df["review_comment_message"].notnull())].head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
306,8e24261a7e58791d10cb1bf9da94df5c,64a254d30eed42cd0e6c36dddb88adf0,unavailable,2017-11-16 15:09:28,2017-11-16 15:26:57,NaN,NaN,2017-12-05 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,41fc647b8c6bd979b1b6364b60471b50,89288,sao bento do sul,SC,84.00,5.0,credit_card,0.0,0.00,1.0,Anunciam um produto que não tem em estoque e a...,2017-12-07 00:00:00,2017-12-11 10:37:57,NaN,NaN,NaN,NaN,-26.232194,-49.411882,sao bento do sul,SC
791,37553832a3a89c9b2db59701c357ca67,7607cd563696c27ede287e515812d528,unavailable,2017-08-14 17:38:02,2017-08-17 00:15:18,NaN,NaN,2017-09-05 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,596ed6d7a35890b3fbac54ec01f69685,2318,sao paulo,SP,132.46,1.0,boleto,0.0,0.00,1.0,Até hoje não recebi meu produto e nem uma resp...,2017-09-10 00:00:00,2017-09-12 17:00:15,NaN,NaN,NaN,NaN,-23.450859,-46.590111,sao paulo,SP
850,d57e15fb07fd180f06ab3926b39edcd2,470b93b3f1cde85550fc74cd3a476c78,unavailable,2018-01-08 19:39:03,2018-01-09 07:26:08,NaN,NaN,2018-02-06 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,2349bbb558908e0955e98d47dacb7adb,48607,paulo afonso,BA,134.38,1.0,boleto,0.0,0.00,1.0,Não compre,2018-02-09 00:00:00,2018-02-09 03:22:43,NaN,NaN,NaN,NaN,-9.395805,-38.218861,paulo afonso,BA
1294,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaN,NaN,NaN,2018-09-12 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,4fa4365000c7090fcb8cad5713c6d3db,1151,sao paulo,SP,0.00,1.0,not_defined,0.0,0.00,1.0,Comprei dois fones de ouvido com valor de R$ 5...,2018-08-28 00:00:00,2018-08-28 18:25:55,NaN,NaN,NaN,NaN,-23.531642,-46.656289,sao paulo,SP
1326,2f634e2cebf8c0283e7ef0989f77d217,7353b0fb8e8d9675e3a704c60ca44ebe,unavailable,2017-09-27 20:55:33,2017-09-28 01:32:50,NaN,NaN,2017-10-27 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,21c933c8dd97d088e64c50988c90ccf5,5017,sao paulo,SP,615.53,12.0,credit_card,0.0,0.00,1.0,"Comprei um perfume Bleu de Chanel , paguei e n...",2017-10-29 00:00:00,2017-10-29 16:40:21,NaN,NaN,NaN,NaN,-23.537114,-46.679678,sao paulo,SP
1802,ee0db22a8e742b752914016708470ec8,aae50600d30bf2efe013ca4c1754ded7,unavailable,2017-08-24 11:04:41,2017-08-24 11:15:11,NaN,NaN,2017-09-18 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,bdc67efa33dd0c3228b91714ac6e363c,23027,rio de janeiro,RJ,167.82,4.0,credit_card,0.0,0.00,1.0,"voces se tornaram corruptos,paguei e ainda nao...",2017-09-20 00:00:00,2017-10-10 21:22:12,NaN,NaN,NaN,NaN,-23.001237,-43.640095,rio de janeiro,RJ
2040,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,canceled,2018-09-20 13:54:16,NaN,NaN,NaN,2018-10-17 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,08642cd329066fe11ec63293f714f2f8,33030,santa luzia,MG,191.46,1.0,voucher,1.0,191.46,2.0,O produto veio com defeito ele não liga não fu...,2018-07-28 00:00:00,2018-07-30 11:06:16,NaN,NaN,NaN,NaN,-19.775876,-43.872971,santa luzia,MG
2074,6ad57aecbae806a7e9cc2cdb6b380711,d31dbd02ac052d662285f6

| Review in Spanish | Review in English|
| --- | --- |
| Anunciam um produto que não tem em estoque e a... | They advertise a product that isn't in stock, and the... |
| Até hoje não recebi meu produto e nem uma resp...	 | To this day, I haven't received my product or even a resp... |
| Não compre | Don't buy it. |
| Comprei dois fones de ouvido com valor de R$ 5...	 | I bought two pairs of headphones costing R$ 5... |
| Comprei um perfume Bleu de Chanel , paguei e n... | I bought a bottle of Bleu de Chanel perfume, paid for it, and... |
| voces se tornaram corruptos,paguei e ainda nao...	 | You’ve become corrupt; I paid, and still haven't... |
| O produto veio com defeito ele não liga não fu...	 | The product arrived defective; it doesn't turn on, it doesn't wo... |
| Estou esperando até agora a entrega do produto... | I'm still waiting for the product to be delivered... |
| Razoável		 | Reasonable |
| Embora não tenha recebido a mercadoria por não...	 | Although I did not receive the merchandise because... |

In [47]:
print(f"Total Unique Order ID with this problem: {df[(df["order_status"]!="delivered")&(df["seller_id"].isnull())]["order_id"].nunique()}")
print(f"Total Percentage of unique id for this missing value: {(df[(df["order_status"]!="delivered")&(df["seller_id"].isnull())]["order_id"].nunique()/len(df[df["order_status"]!="delivered"]))*100:.2f}%")

Total Unique Order ID with this problem: 775
Total Percentage of unique id for this missing value: 24.01%


An investigation into the 775 records missing seller information reveals a significant correlation with critical order failure. A sentiment analysis of the associated customer reviews identifies recurring themes of unresponsiveness, undelivered products, and an inability to secure refunds. These missing values appear to represent high-risk operational anomalies. Such as merchant churn, systemic transaction errors, or platform-enforced account terminations, rather than simple data entry omissions. Essentially, these records document a comprehensive breakdown within the customer fulfillment loop.  
  
Furthermore, as these records constitute 24.01% of the data for non-delivered order statuses, they will be retained in the dataset. Dropping this information would lead to a substantial loss of insight into these specific operational failures and their impact on the platform.

#### **3.1.2.7 Fixing Product Information**

In [48]:
print(f"Missing Data")
print(f"Total Rows  = {df.shape[0]}")
display(df.isnull().sum().to_frame().T)

Missing Data
Total Rows  = 113398


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,146,1966,3221,0,775,775,775,775,775,775,2377,2377,2377,2377,793,793,793,793,2401,775,775,775,0,0,0,0,0,0,0,0,0,961,65493,961,961,775,775,775,775,0,0,0,0


While these missing values could be interpreted as a failure by the seller to provide product information, a discrepancy exists between the total number of null values in `product_category_name` and `product_category_name_english`. Although the latter is intended to be a direct translation of the former, the variance in missing counts indicates that certain product categories may have remained untranslated in the dataset.

In [49]:
df[(df["product_category_name_english"].isnull())&(df["product_category_name"].notnull())]["product_category_name"].unique()

<ArrowStringArray>
['portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer']
Length: 2, dtype: str

| Product Category in Spanish | Product Category in English |
|---|---|
| portateis_cozinha_e_preparadores_de_alimentos | kitchen_appliances_and_food_prep |
| pc_gamer | gaming_pc |

Observation of the dataset identifies two product categories that lack an English translation. To ensure consistency, we will update the missing values in the `product_category_name_english` field by applying the appropriate translations for these specific records.

In [50]:
condition = df["product_category_name"] == "portateis_cozinha_e_preparadores_de_alimentos"
df.loc[condition, "product_category_name_english"] = "kitchen_appliances_and_food_prep"

condition = df["product_category_name"] == "pc_gamer"
df.loc[condition, "product_category_name_english"] = "gaming_pc"

In [51]:
print(f"Missing Data")
print(f"Total Rows  = {df.shape[0]}")
display(df.isnull().sum().to_frame().T)

Missing Data
Total Rows  = 113398


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,146,1966,3221,0,775,775,775,775,775,775,2377,2377,2377,2377,793,793,793,793,2377,775,775,775,0,0,0,0,0,0,0,0,0,961,65493,961,961,775,775,775,775,0,0,0,0


#### **3.1.2.7 Fixing Review Information**

In [52]:
print(f"Missing Data")
print(f"Total Rows  = {df.shape[0]}")
display(df.isnull().sum().to_frame().T)

Missing Data
Total Rows  = 113398


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,146,1966,3221,0,775,775,775,775,775,775,2377,2377,2377,2377,793,793,793,793,2377,775,775,775,0,0,0,0,0,0,0,0,0,961,65493,961,961,775,775,775,775,0,0,0,0


These missing values can be considered structurally valid, as providing customer feedback or a product review is typically optional in e-commerce transactions. However, to rule out potential data logging errors, we will cross-reference the missing entries in `review_creation_date` and `review_answer_timestamp` against the `order_id` values associated with `review_score`. This verification will ensure that the lack of timestamps does not conflict with existing rating data.

In [53]:
df[(df["review_score"].isnull())&(df["review_creation_date"].isnull())&(df["review_answer_timestamp"].isnull())].isnull().sum().to_frame().T

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_installments,payment_type,n_vouchers,voucher_value,review_score,review_comment_message,review_creation_date,review_answer_timestamp,seller_lat,seller_lng,seller_geo_city,seller_geo_state,customer_lat,customer_lng,customer_geo_city,customer_geo_state
0,0,0,0,0,5,38,134,0,19,19,19,19,19,19,33,33,33,33,19,19,19,19,33,19,19,19,0,0,0,0,0,0,0,0,0,961,961,961,961,19,19,19,19,0,0,0,0


The cross-reference check confirmed that the 961 missing values in `review_creation_date` and `review_answer_timestamp` correspond exactly to missing entries in `review_score`. All three fields are null together, never independently, meaning these are genuinely un-reviewed orders, not a logging error where a score exists without its timestamp (or vice versa).

### **3.1.3 Data Types**

In [54]:
df.info()

<class 'pandas.DataFrame'>
Index: 113398 entries, 0 to 113424
Data columns (total 47 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       113398 non-null  str    
 1   customer_id                    113398 non-null  str    
 2   order_status                   113398 non-null  str    
 3   order_purchase_timestamp       113398 non-null  str    
 4   order_approved_at              113252 non-null  str    
 5   order_delivered_carrier_date   111432 non-null  str    
 6   order_delivered_customer_date  110177 non-null  str    
 7   order_estimated_delivery_date  113398 non-null  str    
 8   order_item_id                  112623 non-null  float64
 9   product_id                     112623 non-null  str    
 10  seller_id                      112623 non-null  str    
 11  shipping_limit_date            112623 non-null  str    
 12  price                          112623 non-null

An initial review of the dataset's schemas indicates that most column data types are correctly assigned. However, several critical temporal fields (`order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date`, and `shipping_limit_date`) are currently formatted as **strings** rather than **datetime** objects. To ensure accurate time-series analysis and proper data manipulation, these columns must be standardized to the correct timestamp format.

In [55]:
datetime_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'shipping_limit_date']

df[datetime_cols] = df[datetime_cols].apply(pd.to_datetime)

In [56]:
df.info()

<class 'pandas.DataFrame'>
Index: 113398 entries, 0 to 113424
Data columns (total 47 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113398 non-null  str           
 1   customer_id                    113398 non-null  str           
 2   order_status                   113398 non-null  str           
 3   order_purchase_timestamp       113398 non-null  datetime64[us]
 4   order_approved_at              113252 non-null  datetime64[us]
 5   order_delivered_carrier_date   111432 non-null  datetime64[us]
 6   order_delivered_customer_date  110177 non-null  datetime64[us]
 7   order_estimated_delivery_date  113398 non-null  datetime64[us]
 8   order_item_id                  112623 non-null  float64       
 9   product_id                     112623 non-null  str           
 10  seller_id                      112623 non-null  str           
 11  shipping_limit_d

### **3.1.4 Uniformity or Standarization Data in Columns**

Beyond missing values and data types, categorical and text columns can also contain inconsistencies that does not show up as nulls (typos, alternate spellings, or duplicate categories that should really be treated as one). The columns below are checked specifically for this: `product_category_name_english`, `payment_type`, `seller_geo_city`, `seller_geo_state`, `customer_geo_city`, and `customer_geo_state`.

#### **3.1.4.1 `product_category_name_english`**

In [57]:
df['product_category_name_english'].unique()

<ArrowStringArray>
[                             'housewares',
                               'perfumery',
                                    'auto',
                                'pet_shop',
                              'stationery',
                                       nan,
                         'furniture_decor',
                        'office_furniture',
                            'garden_tools',
                   'computers_accessories',
                          'bed_bath_table',
                                    'toys',
         'construction_tools_construction',
                               'telephony',
                           'health_beauty',
                             'electronics',
                                    'baby',
                              'cool_stuff',
                           'watches_gifts',
                        'air_conditioning',
                          'sports_leisure',
                  'books_general_interest',
             

The unique values in the product category columns reveal several naming inconsistencies that require standardization to ensure accurate grouping and analysis. These issues primarily fall into two categories: spelling errors and split categorical definitions.  
- **Typographical Errors**: Certain labels contain obvious spelling mistakes, such as `home_confort` (should be `home_comfort`) and `fashio_female_clothing` (should be `fashion_female_clothing`). These errors create redundant categories that fragment the data.  

- **Split Categories**: Some products are divided into numbered versions, such as `home_appliances_2` and `home_comfort_2`. These represent sub-categories that are functionally identical to their primary counterparts for the purpose of high-level analysis and should be merged to maintain categorical integrity.

In [58]:
condition = df["product_category_name_english"] == "home_confort"
df.loc[condition, "product_category_name_english"] = "home_comfort"

condition = df["product_category_name_english"] == "fashio_female_clothing"
df.loc[condition, "product_category_name_english"] = "fashion_female_clothing"

condition = df["product_category_name_english"] == "home_appliances_2"
df.loc[condition, "product_category_name_english"] = "home_appliances"

condition = df["product_category_name_english"] == "home_comfort_2"
df.loc[condition, "product_category_name_english"] = "home_comfort"

#### **3.1.4.2 `payment_type`**

In [59]:
df["payment_type"].unique()

<ArrowStringArray>
['credit_card', 'boleto', 'debit_card', 'voucher', 'not_defined']
Length: 5, dtype: str

`payment_type` is largely uniform, no spelling inconsistencies or split categories were found. However, `not_defined`, present on a very small number of rows. Since it does not describe an actual payment method the way `credit_card`, `boleto`, `debit_card`, and `voucher` do, it functions more like a missing value than a genuine fifth category. Given how few rows it affects, it is left as is for now.

#### **3.1.4.3 `seller_geo_state` and `customer_geo_state`**

Brazil has exactly 27 official UF (state) codes. Any value outside this set is an error, not a "unique" legitimate value.

In [60]:
valid_uf = {
    'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG','PA','PB',
    'PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'
}

state_cols = ["seller_geo_state", "customer_geo_state"]

for c in state_cols:
    vals = set(df[c].dropna().unique())
    invalid = vals - valid_uf
    print(f"{c}: {len(vals)} distinct values, invalid codes -> {invalid or 'none'}")

seller_geo_state: 22 distinct values, invalid codes -> none
customer_geo_state: 27 distinct values, invalid codes -> none


The provided analysis concludes that all **Unidade da Federação (UF)** codes within the dataset strictly adhere to the 27 official Brazilian state and federal district codes. This validation ensures that geographic data is standardized and free from erroneous or unrecognized regional identifiers.

#### **3.1.4.4 `seller_geo_city` and `customer_geo_city`**

##### **3.1.4.4.1 Character-level anomalies (city columns)**

In [61]:
def char_report(col):
    s = df[col].dropna().astype(str)
    non_ascii = s[s.str.contains(r'[^\x00-\x7f]')]
    digits = s[s.str.contains(r'\d')]
    special = s[s.str.contains(r"[^a-zA-Z\s\-']")]
    return {
        'column': col,
        'non_ascii_n': len(non_ascii),
        'digit_n': len(digits),
        'digit_examples': digits.unique()[:5].tolist(),
        'special_punct_n': len(special),
    }

city_cols = ["seller_geo_city", "customer_geo_city"]

pd.DataFrame([char_report(c) for c in city_cols])

,column,non_ascii_n,digit_n,digit_examples,special_punct_n
0,seller_geo_city,442,0,[],451
1,customer_geo_city,195,2,"[Santa Maria (QR 300), quilometro 14 do mutum]",238


##### **3.1.4.4.2 Case-only duplicates, *within (state, city)***

Same city, same state, different capitalization only (e.g. `sao paulo` vs `Sao Paulo`).

In [62]:
def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))

def case_dupes(city_col, state_col):
    sub = df[[city_col, state_col]].dropna().astype(str)
    sub['_norm'] = sub[city_col].str.lower().str.strip()
    grp = sub.groupby([state_col, '_norm'])[city_col].agg(lambda x: sorted(set(x)))
    dupes = grp[grp.apply(len) > 1]
    return dupes

state_cols = ['seller_geo_state', 'customer_geo_state']
pairs = [('seller_geo_city', 'seller_geo_state'), ('customer_geo_city', 'customer_geo_state')]

for city_col, state_col in pairs:
    d = case_dupes(city_col, state_col)
    print(f"{city_col}: {len(d)} (state, city) groups with case-only variants")
    if len(d):
        display(d.head(10))

seller_geo_city: 0 (state, city) groups with case-only variants
customer_geo_city: 51 (state, city) groups with case-only variants


customer_geo_state  _norm            
AL                  campo alegre                   [Campo Alegre, campo alegre]
BA                  candeias                               [Candeias, candeias]
                    euclides da cunha    [Euclides da Cunha, euclides da cunha]
                    lauro de freitas       [Lauro de Freitas, lauro de freitas]
                    salvador                               [Salvador, salvador]
CE                  paraipaba                            [Paraipaba, paraipaba]
                    uruburetama                      [Uruburetama, uruburetama]
ES                  aracruz                                  [Aracruz, aracruz]
GO                  luziânia                               [Luziânia, luziânia]
                    novo gama                            [Novo Gama, novo gama]
Name: customer_geo_city, dtype: object

##### **3.1.4.4.3  Accent + case duplicates, *within (state, city)***

Capitalization alone does not catch everything, accented characters (e.g. `São Paulo` vs. `Sao Paulo`) create the same kind of duplicate. This check normalizes for both at once. strip accents and lowercase, then group by `(state, normalized_city)` to find every city that is really the same place, just spelled differently across rows.

In [63]:
def accent_case_dupes(city_col, state_col):
    sub = df[[city_col, state_col]].dropna().astype(str)
    sub['_norm'] = sub[city_col].apply(lambda x: strip_accents(x).lower().strip())
    grp = sub.groupby([state_col, '_norm'])[city_col].agg(lambda x: sorted(set(x)))
    dupes = grp[grp.apply(len) > 1]
    return dupes.sort_values(key=lambda s: s.apply(len), ascending=False)

accent_dupe_results = {}
for city_col, state_col in pairs:
    d = accent_case_dupes(city_col, state_col)
    accent_dupe_results[city_col] = d
    print(f"{city_col}: {len(d)} (state, city) groups collapse once accents+case are normalized")
    if len(d):
        display(d.head(15))

seller_geo_city: 4 (state, city) groups collapse once accents+case are normalized


seller_geo_state  _norm          
MG                pocos de caldas    [Poços de Caldas, pocos de caldas]
SP                aruja                                  [Arujá, aruja]
                  jaguariuna                   [jaguariuna, jaguariúna]
                  sao paulo                      [sao paulo, são paulo]
Name: seller_geo_city, dtype: object

customer_geo_city: 78 (state, city) groups collapse once accents+case are normalized


customer_geo_state  _norm                  
SP                  sao paulo                                  [São Paulo, sao paulo, são paulo]
GO                  luziania                                      [Luziânia, luziania, luziânia]
RS                  estancia velha              [Estância Velha, estancia velha, estância velha]
AL                  campo alegre                                    [Campo Alegre, campo alegre]
RJ                  bom jesus do itabapoana    [Bom Jesus do Itabapoana, bom jesus do itabapo...
                    nova friburgo                                 [Nova Friburgo, nova friburgo]
                    mage                                                            [Magé, mage]
                    itaocara                                                [Itaocara, itaocara]
                    cordeiro                                                [Cordeiro, cordeiro]
                    carmo                                                         [

##### **3.1.4.4.4 Structural fragmentation**

Some rows append a neighborhood or administrative sub-district in parentheses to the city name (e.g. `Brasília (Lago Norte / Taquari)`, `Santa Maria (QR 300)`). These are not different cities, they are the same city but fragmented into extra unique values by the added detail. The check below is grouped by state specifically to avoid a false merge. Two different real cities in different states could coincidentally share a base name once the parenthetical part is stripped, and grouping by state first prevents that from happening.

In [64]:
def paren_fragmentation(city_col, state_col):
    sub = df[[city_col, state_col]].dropna().astype(str)
    has_paren = sub[sub[city_col].str.contains(r'\(')]
    if has_paren.empty:
        return pd.DataFrame(), has_paren
    base = has_paren[city_col].str.replace(r'\s*\(.*\)', '', regex=True).str.strip()
    tmp = has_paren.assign(_base=base)
    grouped = tmp.groupby([state_col, '_base'])[city_col].agg(lambda x: sorted(set(x)))
    return grouped, has_paren

for city_col, state_col in pairs:
    grouped, raw = paren_fragmentation(city_col, state_col)
    print(f"{city_col}: {len(raw)} rows carry a parenthetical suffix, "
          f"collapsing into {grouped.index.get_level_values('_base').nunique() if len(grouped) else 0} base (state, city) groups")
    if len(grouped):
        display(grouped.head(15))

seller_geo_city: 190 rows carry a parenthetical suffix, collapsing into 4 base (state, city) groups


seller_geo_state  _base       
DF                Brasília        [Brasília (Condomínio Residencial Santa Maria)...
PR                Curitiba                     [Curitiba (Santa Felicidade Sector)]
RS                Porto Alegre            [Porto Alegre (Tristeza / Vila Assunção)]
SP                São Paulo                          [São Paulo (Jardim Cachoeira)]
Name: seller_geo_city, dtype: object

customer_geo_city: 151 rows carry a parenthetical suffix, collapsing into 18 base (state, city) groups


customer_geo_state  _base                
BA                  Salvador                                             [Salvador (Subúrbio)]
DF                  Brasília                 [Brasília (Asa Norte - SCRN), Brasília (Asa No...
                    Ceilândia                [Ceilândia (P Sul), Ceilândia (QNM Sector), Ce...
                    Gama                     [Gama (Setor Central), Gama (Setor Leste), Gam...
                    Planaltina               [Planaltina (Arapoanga), Planaltina (Estâncias...
                    Samambaia                    [Samambaia (Expansão), Samambaia (QN Sector)]
                    Santa Maria              [Santa Maria (CL Sector), Santa Maria (Norte),...
                    Águas Claras             [Águas Claras (Areal), Águas Claras (SHA Secto...
GO                  Novo Gama                                            [Novo Gama (Lunabel)]
MG                  Cabeceira Grande                    [Cabeceira Grande (Palmital de Minas)]
        

##### **3.1.4.4.5 Build a suggested canonicalization mapping**

Combining everything found above (case, accents, and parenthetical suffixes) into one rule: normalize every city name to lowercase, ASCII-only (accents stripped), with any parenthetical suffix removed. This single rule is what collapses all three types of duplication identified above into one canonical spelling per city.

In [65]:
def build_mapping(city_col, state_col):
    sub = df[[city_col, state_col]].dropna().astype(str).copy()
    sub['_base'] = sub[city_col].str.replace(r'\s*\(.*\)', '', regex=True).str.strip()
    sub['canonical'] = sub['_base'].apply(lambda x: strip_accents(x).lower().strip())

    changed = sub[sub[city_col] != sub['canonical']][[city_col, state_col, 'canonical']].drop_duplicates()
    changed = changed.rename(columns={city_col: 'raw_value', state_col: 'state'})
    changed['column'] = city_col
    return changed[['column', 'state', 'raw_value', 'canonical']]

mapping = pd.concat([build_mapping(city_col, state_col) for city_col, state_col in pairs], ignore_index=True)
mapping = mapping.sort_values(['column', 'state', 'raw_value']).reset_index(drop=True)
print(f"{len(mapping)} raw values flagged with a suggested canonical replacement")
mapping.head(30)

184 raw values flagged with a suggested canonical replacement


,column,state,raw_value,canonical
0,customer_geo_city,AL,Campo Alegre,campo alegre
1,customer_geo_city,AL,são miguel dos campos,sao miguel dos campos
2,customer_geo_city,BA,Camaçari,camacari
3,customer_geo_city,BA,Candeias,candeias
4,customer_geo_city,BA,Euclides da Cunha,euclides da cunha
5,customer_geo_city,BA,Lauro de Freitas,lauro de freitas
6,customer_geo_city,BA,Salvador,salvador
7,customer_geo_city,BA,Salvador (Subúrbio),salvador
8,customer_geo_city,BA,Santo Estêvão,santo estevao
9,customer_geo_city,BA,caém,caem


##### **3.1.4.4.5 Apply the mapping**

In [66]:
lookup_seller = dict(zip(mapping.loc[mapping['column']=='seller_geo_city','raw_value'],
                          mapping.loc[mapping['column']=='seller_geo_city','canonical']))
lookup_customer = dict(zip(mapping.loc[mapping['column']=='customer_geo_city','raw_value'],
                            mapping.loc[mapping['column']=='customer_geo_city','canonical']))
df['seller_geo_city'] = df['seller_geo_city'].replace(lookup_seller)
df['customer_geo_city'] = df['customer_geo_city'].replace(lookup_customer)

## **3.2 Logical Validity**

### **3.2.1 `order_purchase_timestamp`**

`order_purchase_timestamp` is the anchor point of an order's entire lifecycle. Every later stage timestamp (approval, carrier handoff, delivery, the estimated delivery date, and the seller's shipping deadline) should logically occur **after** the purchase itself, since none of those events can happen before a customer actually places the order. We check all five downstream date columns against `order_purchase_timestamp`. Any row where a later stage date falls **before** the purchase timestamp indicates a data entry or export error, not a real sequence of events.

In [67]:
unique_orders = set()
unique_orders.update(df[df["order_approved_at"]<df["order_purchase_timestamp"]]["order_id"].unique())
unique_orders.update(df[df["order_delivered_carrier_date"]<df["order_purchase_timestamp"]]["order_id"].unique())
unique_orders.update(df[df["order_delivered_customer_date"]<df["order_purchase_timestamp"]]["order_id"].unique())
unique_orders.update(df[df["order_estimated_delivery_date"]<df["order_purchase_timestamp"]]["order_id"].unique())
unique_orders.update(df[df["shipping_limit_date"]<df["order_purchase_timestamp"]]["order_id"].unique())

print(f"Total order that is error based on `order_approved_at`: {df[df["order_approved_at"]<df["order_purchase_timestamp"]]["order_id"].nunique()}")
print(f"Total order that is error based on `order_delivered_carrier_date`: {df[df["order_delivered_carrier_date"]<df["order_purchase_timestamp"]]["order_id"].nunique()}")
print(f"Total order that is error based on `order_delivered_customer_date`: {df[df["order_delivered_customer_date"]<df["order_purchase_timestamp"]]["order_id"].nunique()}")
print(f"Total order that is error based on `order_estimated_delivery_date`: {df[df["order_estimated_delivery_date"]<df["order_purchase_timestamp"]]["order_id"].nunique()}")
print(f"Total order that is error based on `shipping_limit_date`: {df[df["shipping_limit_date"]<df["order_purchase_timestamp"]]["order_id"].nunique()}")

Total order that is error based on `order_approved_at`: 0
Total order that is error based on `order_delivered_carrier_date`: 166
Total order that is error based on `order_delivered_customer_date`: 0
Total order that is error based on `order_estimated_delivery_date`: 0
Total order that is error based on `shipping_limit_date`: 0


### **3.2.2 `order_approved_at`**

`order_approved_at` marks payment confirmation, the next checkpoint after purchase. From this point on, the same logic applies one step later in the chain: carrier handoff, customer delivery, the estimated delivery date, and the shipping deadline should all occur at or after approval. We repeat the same before/after check here, this time comparing every remaining downstream date against `order_approved_at`.

In [68]:
unique_orders.update(df[df["order_delivered_carrier_date"]<df["order_approved_at"]]["order_id"].unique())
unique_orders.update(df[df["order_delivered_customer_date"]<df["order_approved_at"]]["order_id"].unique())
unique_orders.update(df[df["order_estimated_delivery_date"]<df["order_approved_at"]]["order_id"].unique())
unique_orders.update(df[df["shipping_limit_date"]<df["order_approved_at"]]["order_id"].unique())

print(f"Total order that is error based on `order_delivered_carrier_date`: {df[df["order_delivered_carrier_date"]<df["order_approved_at"]]["order_id"].nunique()}")
print(f"Total order that is error based on `order_delivered_customer_date`: {df[df["order_delivered_customer_date"]<df["order_approved_at"]]["order_id"].nunique()}")
print(f"Total order that is error based on `order_estimated_delivery_date`: {df[df["order_estimated_delivery_date"]<df["order_approved_at"]]["order_id"].nunique()}")
print(f"Total order that is error based on `shipping_limit_date`: {df[df["shipping_limit_date"]<df["order_approved_at"]]["order_id"].nunique()}")

Total order that is error based on `order_delivered_carrier_date`: 1359
Total order that is error based on `order_delivered_customer_date`: 61
Total order that is error based on `order_estimated_delivery_date`: 12
Total order that is error based on `shipping_limit_date`: 116


### **3.2.3 `order_delivered_carrier_date`**

The last pairwise check in the fulfillment sequence: an order cannot be delivered to the customer before it was ever handed to the carrier in the first place. We check `order_delivered_customer_date` against `order_delivered_carrier_date` specifically for this .Any row where the customer delivery date comes first is a logical error.

In [69]:
unique_orders.update(df[df["order_delivered_customer_date"]<df["order_delivered_carrier_date"]]["order_id"].unique())

print(f"Total order that is error based on `order_delivered_customer_date`: {df[df["order_delivered_customer_date"]<df["order_delivered_carrier_date"]]["order_id"].nunique()}")

Total order that is error based on `order_delivered_customer_date`: 23


### **3.2.4 Total Logical Fallacy**

Having checked each stage of the order lifecycle individually above, we now combine every logically inconsistent order found across all of those checks into a single set, so we can see the true combined scope of the problem and decide how to handle it.

In [70]:
print(f"Total Unique Order ID: {df[df['order_id'].isin(unique_orders)]["order_id"].nunique()}")
print(f"Total Unique Order ID on Delivered Order: {df[(df['order_id'].isin(unique_orders))&(df["order_status"]=="delivered")]["order_id"].nunique()}")
print(f"Total Unique Order ID on Order Status other than Delivered: {df[(df['order_id'].isin(unique_orders))&(df["order_status"]!="delivered")]["order_id"].nunique()}")

Total Unique Order ID: 1400
Total Unique Order ID on Delivered Order: 1384
Total Unique Order ID on Order Status other than Delivered: 16


In [71]:
print(f"Total Percetage Unique Order ID on Delivered Order: {(df[(df['order_id'].isin(unique_orders))&(df["order_status"]=="delivered")]["order_id"].nunique()/len(df[df["order_status"]=="delivered"]))*100:.2f}%")
print(f"Total Percentage Unique Order ID on Order Status other than Delivered: {(df[(df['order_id'].isin(unique_orders))&(df["order_status"]!="delivered")]["order_id"].nunique()/len(df[df["order_status"]!="delivered"]))*100:.2f}%")

Total Percetage Unique Order ID on Delivered Order: 1.26%
Total Percentage Unique Order ID on Order Status other than Delivered: 0.50%


An assessment of the sequential logic from order placement to final customer delivery identified a minimal error rate: **1.26%** for orders with a "delivered" status and **0.5%** for all other statuses. Given the negligible volume of these logical fallacies, we have decided to remove these records from the dataset. This ensures the integrity of the analysis and prevents anomalous data from skewing our strategic insights.

In [72]:
df = df[~df['order_id'].isin(unique_orders)]

## **3.3 Unused Columns**

**1. Removal of Non-Essential Features**
Certain columns in the dataset provide metadata that does not contribute meaningful value to the core analysis of order fulfillment or transaction performance. Specifically, the following features will be removed:
- `product_name_lenght`
- `product_description_lenght`
- `product_photos_qty`  
  
**Rationale**: These attributes focus on product listing aesthetics rather than the transactional lifecycle. In data science workflows, purging irrelevant features enhances model efficiency, reduces computational overhead, and prevents the introduction of "noise" that could otherwise hinder predictive accuracy.  
  
**2. Elimination of Redundant Data**  
To optimize the dataset's structure and ensure a single source of truth, several redundant columns will be dropped:
- `product_category_name`: Retaining only the English translation ensures consistent reporting.
- `customer_city` and `customer_state`
- `seller_city` and `seller_state`  
  
**Rationale**: Following the data merging process, these geographic fields often overlap with other location-based tables (such as geolocation data). Keeping multiple features that measure the same attribute is redundant. By streamlining these columns, we maintain data integrity and ensure that regional performance metrics are derived from the most consistent and accurate sources available

In [73]:
df.drop(columns=['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_category_name', 'customer_city', "customer_state", 'seller_city', 'seller_state'], inplace=True)

## **3.4 Exporting Cleaned Dataset**

In [74]:
df.to_csv(f'../{DATA_DIR}/Cleaned Data/cleaned.csv', index=False)
print('Saved with shape', df.shape)

Saved with shape (111777, 39)
